# Attribution Demo

In this demo, you'll learn how to load models and perform attribution on them using [circuit-tracer](https://github.com/decoderesearch/circuit-tracer).

In [2]:
# Install circuit-tracer from GitHub (not on PyPI)
!pip install git+https://github.com/decoderesearch/circuit-tracer.git
!pip install torch


Looking in indexes: https://pypi.apple.com/simple
  Cloning https://github.com/decoderesearch/circuit-tracer.git to /mnt/tmp/pip-req-build-v1sehug4
  Running command git clone --filter=blob:none --quiet https://github.com/decoderesearch/circuit-tracer.git /mnt/tmp/pip-req-build-v1sehug4
  Resolved https://github.com/decoderesearch/circuit-tracer.git to commit a2e9eb9ab0e7c7a4f033ff4bafa1bb140412d162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in indexes: https://pypi.apple.com/simple


In [3]:
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files

In [4]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")

## Load Model

Load your model and transcoders by name. `model_name` is a HuggingFace / TransformerLens model name.

Supported models:
- `google/gemma-2-2b` with `"gemma"` transcoders (Gemma Scope, lowest L0)
- `meta-llama/Llama-3.2-1B` with `"llama"` transcoders (ReLU skip-transcoders)

For other models, provide your own transcoder config file.

In [5]:
model_name = "google/gemma-2-2b"
transcoder_name = "gemma"
backend = "transformerlens"  # change to 'nnsight' for the nnsight backend
model = ReplacementModel.from_pretrained(
    model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


## Set Attribution Arguments

In [6]:
prompts = {
    "dallas-austin": "The capital of state containing Dallas is",
    "paris-france": "The capital of France is",
    "lying-dallas": "You are a lying assistant. The capital of state containing Dallas is",
}

max_n_logits = 10
desired_logit_prob = 0.95
max_feature_nodes = 8192
batch_size = 256
offload = "cpu"  # 'disk', 'cpu', or None (keep on GPU)
verbose = True

## Run Attribution

In [7]:
graph_dir = Path("graphs")
graph_dir.mkdir(exist_ok=True)
graph_file_dir = "./graph_files"
node_threshold = 0.8
edge_threshold = 0.98

for slug, prompt in prompts.items():
    print(f"\n{'='*60}\nAttributing: {slug} — \"{prompt}\"\n{'='*60}")

    graph = attribute(
        prompt=prompt,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        batch_size=batch_size,
        max_feature_nodes=max_feature_nodes,
        offload=offload,
        verbose=verbose,
    )

    graph_path = graph_dir / f"{slug}.pt"
    graph.to_pt(graph_path)
    print(f"Saved graph to {graph_path}")

    create_graph_files(
        graph_or_path=graph_path,
        slug=slug,
        output_path=graph_file_dir,
        node_threshold=node_threshold,
        edge_threshold=edge_threshold,
    )
    print(f"Done: {slug}")

Phase 0: Precomputing activations and vectors



Attributing: dallas-austin — "The capital of state containing Dallas is"


Precomputation completed in 0.41s
Found 6344 active features
Phase 1: Running forward pass
Forward pass completed in 0.08s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.7148
Will include 6344 of 6344 feature nodes
Input vectors built in 1.19s
Phase 3: Computing logit attributions
sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
Logit attributions completed in 0.05s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 6344/6344 [00:00<00:00, 20050.29it/s]
Feature attributions completed in 0.32s
Attribution completed in 4.40s


Saved graph to graphs/dallas-austin.pt


Phase 0: Precomputing activations and vectors
Precomputation completed in 0.09s
Found 4417 active features


...

Saved graph to graphs/paris-france.pt


Phase 0: Precomputing activations and vectors
Precomputation completed in 0.13s
Found 11678 active features


Done: paris-france

Attributing: lying-dallas — "You are a lying assistant. The capital of state containing Dallas is"


Phase 1: Running forward pass
Forward pass completed in 0.08s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.6562
Will include 8192 of 11678 feature nodes
Input vectors built in 1.20s
Phase 3: Computing logit attributions
Logit attributions completed in 0.05s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:01<00:00, 5324.65it/s]
Feature attributions completed in 1.54s
Attribution completed in 5.58s


Saved graph to graphs/lying-dallas.pt
Done: lying-dallas


## Visualize the Graph

Spin up a local server for the interactive frontend.

**If running on a remote server, set up port forwarding so the chosen port is accessible locally.**

Interactions:
- Click nodes to select them
- Ctrl/Cmd+Click to pin/unpin nodes to subgraph
- G+Click to group nodes into a supernode
- G+Click on X to dissolve a supernode

In [9]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8048
server = serve(data_dir="./graph_files/", port=port)

print(f"Open your graph here: http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

Open your graph here: http://localhost:8048/index.html


In [ ]:
# Stop the server when done
# server.stop()

## Optional: Prune Graph Further

You can re-prune with different thresholds to get smaller/larger circuits.

In [ ]:
from circuit_tracer.graph import prune_graph

prune_graph(graph, node_threshold=0.7, edge_threshold=0.95)